In [1]:
import os
import pandas as pd

In [9]:
# file paths
home = '/store/carroll/sbgplants/'
ref = os.path.join(home, 'schema')
raw = os.path.join(home, 'data', 'raw')

doi = os.path.join(raw, '10.15485.1618130') # Locations, metadata, and species cover from field sampling survey associated with NEON AOP survey, East River, CO 2018

out_folder = os.path.join(home, 'data', 'out_csv')

table = 'plot'

In [6]:
# load schema and dtype
schema = pd.read_csv(os.path.join(ref, 'sbgplants-schema.csv'))
data_types = pd.read_csv(os.path.join(ref, 'data-types.csv'))

# view relevant schema
schema = schema[schema.table_name==table]
schema

,table_name,column_name,data_type
44,plot,plot_name,character
45,plot,plot_type,USER-DEFINED
46,plot,campaign,character


In [7]:
# view relevant data-types info
# for now we can't do this automatically by column_name, because column_name is not explicitly linked to enum_type in any way yet

data_types = data_types[data_types['Enum Type']=='PLOT_type']
data_types

,Schema,Enum Type,Enum Value
19,sbgplants,PLOT_type,Individual
20,sbgplants,PLOT_type,Plot
21,sbgplants,PLOT_type,Transect


In [8]:
# load relevant reference output tables
campaign = pd.read_csv(os.path.join(out_folder, 'campaign.csv'))
campaign

,campaign_name,primary_funding_agency,data_repository,doi
0,East River 2018,DOE,ESS-DIVE,"10.15485/1618130, 10.15485/1618132, 10.15485/1..."


In [10]:
# load relevant reference raw tables
sample_site = pd.read_csv(os.path.join(doi,'sample_site.csv'))

# derive plot_type from VegetationType
plot_type = {
    'Meadow': 'Plot',
    'Tree': 'Individual',
    'Shrub': 'Individual'
}
sample_site['plot_type'] = sample_site['VegetationType'].map(plot_type)

sample_site

,SamplingArea,Campaign,SampleSiteCode,Month,Day,Year,Longitude,Latitude,EPSG,GPS_source,Elevation_m,VegetationType,FieldVegHeightMax_cm,FieldVegHeightMedian_cm,SoilMoisture_%_1,SoilMoisture_%_2,SoilMoisture_%_3,Foliar_IGSN,plot_type
0,RM,ER18,001-ER18,6,14,2018,38.957229,-106.986098,4326.0,RTK,2908.8,Meadow,21.0,13.0,6.0,6.0,7.0,IER18005A,Plot
1,RM,ER18,002-ER18,6,14,2018,38.957288,-106.986100,4326.0,RTK,2908.8,Meadow,42.0,26.0,4.0,6.0,4.0,IER18005B,Plot
2,RM,ER18,003-ER18,6,14,2018,38.957368,-106.986160,4326.0,RTK,2908.2,Meadow,31.0,17.0,6.0,7.0,5.0,IER18005C,Plot
3,RM,ER18,004-ER18,6,14,2018,38.957477,-106.986105,4326.0,RTK,2908.3,Meadow,55.0,32.0,5.0,7.0,8.0,IER18005D,Plot
4,RM,ER18,005-ER18,6,14,2018,38.957461,-106.986231,4326.0,RTK,2907.8,Meadow,59.0,14.0,7.0,13.0,12.0,IER18005E,Plot
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
472,GS,ER18,474-ER18,7,30,2018,38.868239,-107.042181,4326.0,TrimbleGeoXT,3015.6,Tree,NaN,NaN,NaN,NaN,NaN,IER180056,Individual
473,GS,ER18,475-ER18,7,30,2018,38.867972,-107.041750,4326.0,TrimbleGeoXT,3012.5,Tree,NaN,NaN,NaN,NaN,NaN,IER180057,Individual
474,GS,ER18,476-ER18,7,30,2018,38.867825,-107.041729,4326.0,TrimbleGeoXT,3008.0,Tree,NaN,NaN,NaN,NaN,NaN,IER180058,Individual
475,GS,ER18,477-ER18,7,30,2018,38.866244,-107.041369,4326.0,TrimbleGeoXT,3001.7,Tree,NaN,NaN,NaN,NaN,NaN,IER180049,Individual


In [11]:
# prepare & populate out table
out_table = pd.DataFrame(columns=schema['column_name'].unique())

# pull directly from another existing output table where possible
out_table['plot_name'] = sample_site['SampleSiteCode']
out_table['plot_type'] = sample_site['plot_type']
out_table['campaign'] = campaign.campaign_name[0]

out_table

,plot_name,plot_type,campaign
0,001-ER18,Plot,East River 2018
1,002-ER18,Plot,East River 2018
2,003-ER18,Plot,East River 2018
3,004-ER18,Plot,East River 2018
4,005-ER18,Plot,East River 2018
...,...,...,...
472,474-ER18,Individual,East River 2018
473,475-ER18,Individual,East River 2018
474,476-ER18,Individual,East River 2018
475,477-ER18,Individual,East River 2018


In [12]:
# check data types
out_table.dtypes

plot_name    object
plot_type    object
campaign     object
dtype: object

In [13]:
# export table
fp_out = os.path.join(out_folder, f'{table}.csv')
out_table.to_csv(fp_out, index=False)